# Batch Feature Extractor (Voronoi Segmentation)
This notebook isolates specific feature point clouds using Nearest-Neighbor (Voronoi) segmentation, calculates their exact Chamfer Distance noise without distance truncation, and saves them to disk to serve as the training dataset for PointNet.

In [1]:
# ==========================================
# 1. INTERACTIVE VISUALIZATION & DEBUGGING
# ==========================================
# Run this cell on a single viewpoint to visually verify that the Voronoi 
# segmentation correctly maps noisy points to their corresponding features.
import math
import os
import glob
import copy
import numpy as np
import open3d as o3d

WORKPIECE = "TH0011AV"
VIEWPOINT_IDX = 382
EXPERIMENT = "test_9_simulation_3"
NUMBER_OF_POINTS = 5000

print(f"Loading {WORKPIECE} - Viewpoint {VIEWPOINT_IDX} for interactive validation...")

PROCESSED_DIR = f"processed_data/{EXPERIMENT}/{WORKPIECE}"
WORKPIECE_DIR = f"workpiece/{WORKPIECE}"
noisy_pcd_path = os.path.join(PROCESSED_DIR, f"viewpoint_simulated_noise_{VIEWPOINT_IDX}.pcd")

def default_visualization(geometries, window_name="Default Visualization", zoom=1.0):
    azimuth_deg = -45
    elevation_deg = -135
    az = math.radians(azimuth_deg)
    el = math.radians(elevation_deg)
    front = np.array([math.cos(el) * math.cos(az), math.cos(el) * math.sin(az), math.sin(el)])
    front = -front
    if isinstance(geometries, list) and len(geometries) > 0:
        lookat = geometries[0].get_center()
    else:
        lookat = [0, 0, 0]
    up = [0, 0, 1]
    o3d.visualization.draw_geometries(geometries, window_name=window_name, width=1024, height=768, lookat=lookat, up=up, front=front, zoom=zoom)

try:
    # 1. Load full workpiece CAD model
    workpiece_mesh = o3d.io.read_triangle_mesh(os.path.join(WORKPIECE_DIR, "workpiece.stl"))
    workpiece_mesh.compute_vertex_normals()
    # Generate more points so the background is dense enough
    workpiece_pcd = workpiece_mesh.sample_points_poisson_disk(number_of_points=NUMBER_OF_POINTS * 4)

    # 2. Load all target features
    surface_files = glob.glob(os.path.join(WORKPIECE_DIR, "surface*.stl"))
    feature_pcds = []
    feature_names = []
    for f in surface_files:
        mesh = o3d.io.read_triangle_mesh(f)
        mesh.compute_vertex_normals()
        pcd = mesh.sample_points_poisson_disk(number_of_points=NUMBER_OF_POINTS)
        feature_pcds.append(pcd)
        feature_names.append(os.path.basename(f))

    # 3. Mathematically subtract Target Features from Full Workpiece to generate Background
    dists_to_features = np.ones(len(workpiece_pcd.points)) * 999.0
    for fpcd in feature_pcds:
        dists = np.asarray(workpiece_pcd.compute_point_cloud_distance(fpcd))
        dists_to_features = np.minimum(dists_to_features, dists)
    
    background_indices = np.where(dists_to_features > 2.0)[0]
    background_pcd = workpiece_pcd.select_by_index(background_indices)
    
    # Define our multi-class array: Target features first, then background
    classes_pcd = feature_pcds.copy()
    class_names = feature_names.copy()
    if len(background_pcd.points) > 0:
        classes_pcd.append(background_pcd)
        class_names.append("Background (Outer Wall)")


    # 4. Load NOISY simulated PCD
    noisy_pcd = o3d.io.read_point_cloud(noisy_pcd_path)

    # 5. Voronoi Segmentation (Nearest Neighbor Competition)
    print("\nRunning Voronoi Segmentation...")
    dists = np.zeros((len(noisy_pcd.points), len(classes_pcd)))
    for i, cls_pcd in enumerate(classes_pcd):
        dists[:, i] = np.asarray(noisy_pcd.compute_point_cloud_distance(cls_pcd))
    
    # Every point is assigned the index of the class it is closest to
    assignments = np.argmin(dists, axis=1)

    # 6. Visualization
    geometries = []
    colors = [[1, 0, 0], [0, 1, 0], [0, 0, 1], [1, 1, 0], [1, 0, 1], [0, 1, 1]]
    print("\nSegmentation Results:")
    for i, name in enumerate(class_names):
        idx = np.where(assignments == i)[0]
        assigned_pcd = noisy_pcd.select_by_index(idx)
        
        if "Background" in name:
            color = [0.5, 0.5, 0.5] # Gray for outer wall
            print(f"  [Ignored] {name} -> Absorbed {len(idx)} points (Color: Gray)")
        else:
            color = colors[i % len(colors)]
            print(f"  [Target] {name} -> Captured {len(idx)} points (Color: {color})")
            
        assigned_pcd.paint_uniform_color(color)
        geometries.append(assigned_pcd)
        
        # Add reference feature next to it (shifted left by 100mm)
        ref_pcd = copy.deepcopy(classes_pcd[i])
        ref_pcd.paint_uniform_color(color)
        ref_pcd.translate([-100, 0, 0])
        geometries.append(ref_pcd)

    default_visualization(geometries, window_name="Voronoi Segmentation of Noisy Point Cloud")

except Exception as e:
    print(f"Error loading or visualizing point clouds: {e}")


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
Loading TH0011AV - Viewpoint 382 for interactive validation...

Running Voronoi Segmentation...

Segmentation Results:
  [Ignored] Background (Outer Wall) -> Absorbed 4249 points (Color: Gray)


In [2]:
# ==========================================
# 2. BATCH PROCESSING LOOP
# ==========================================
import os
import glob
import pandas as pd
import numpy as np
import open3d as o3d

# WORKPIECES = ["TH0011AV"]
WORKPIECES = ["TH0011AV", "TH0012AV", "TH0021AV", "TH0022AV", "TH0031AV", "TH0032AV", "TH0041AV", "TH0042AV", "TH0051AV", "TH0052AV", "TH0061AV", "TH0062AV", "TH0071AV", "TH0072AV"]
EXPERIMENT = "test_9_simulation_3"
DATASET_NAME = EXPERIMENT
NUMBER_OF_POINTS = 5000

results = []

for workpiece in WORKPIECES:
    print(f"\n==========================================")
    print(f"ANALYZING & EXTRACTING SURFACES: {workpiece}")
    print(f"==========================================")
    
    PROCESSED_DIR = f"processed_data/{EXPERIMENT}/{workpiece}"
    SIM_DIR = f"viewpoints_candidate/testing_data/{EXPERIMENT}/{workpiece}"
    WORKPIECE_DIR = f"workpiece/{workpiece}"
    
    EXPORT_DIR = PROCESSED_DIR
    os.makedirs(EXPORT_DIR, exist_ok=True)
    
    if not os.path.exists(PROCESSED_DIR):
        print(f"Skipping {workpiece} - No processed data found.")
        continue
        
    # 1. Load full workpiece CAD model
    workpiece_mesh = o3d.io.read_triangle_mesh(os.path.join(WORKPIECE_DIR, "workpiece.stl"))
    workpiece_mesh.compute_vertex_normals()
    workpiece_pcd = workpiece_mesh.sample_points_poisson_disk(number_of_points=NUMBER_OF_POINTS * 4)
        
    # 2. Load all target features
    surface_files = glob.glob(os.path.join(WORKPIECE_DIR, "surface*.stl"))
    if not surface_files:
        print(f"No surface features found for {workpiece}.")
        continue
        
    feature_pcds = []
    feature_names = []
    for f in surface_files:
        mesh = o3d.io.read_triangle_mesh(f)
        mesh.compute_vertex_normals()
        pcd = mesh.sample_points_poisson_disk(number_of_points=NUMBER_OF_POINTS)
        feature_pcds.append(pcd)
        feature_names.append(os.path.basename(f))
        
    # 3. Generate Background PCD
    dists_to_features = np.ones(len(workpiece_pcd.points)) * 999.0
    for fpcd in feature_pcds:
        dists = np.asarray(workpiece_pcd.compute_point_cloud_distance(fpcd))
        dists_to_features = np.minimum(dists_to_features, dists)
    
    background_indices = np.where(dists_to_features > 2.0)[0]
    background_pcd = workpiece_pcd.select_by_index(background_indices)
    
    classes_pcd = feature_pcds.copy()
    class_names = feature_names.copy()
    if len(background_pcd.points) > 0:
        classes_pcd.append(background_pcd)
        class_names.append("Background (Outer Wall)")

        
    # Find all simulated point clouds
    SIMULATION_OUTPUT_DIR = f"simulation/{EXPERIMENT}/{workpiece}"
    pcd_files = glob.glob(os.path.join(SIMULATION_OUTPUT_DIR, "viewpoint_simulated_noise_*.pcd"))
    if not pcd_files:
        pcd_files = glob.glob(os.path.join(PROCESSED_DIR, "viewpoint_simulated_noise_*.pcd"))
    pcd_files = [f for f in pcd_files if "_surface" not in f]
    
    # Sort files numerically instead of lexicographically
    import re
    def extract_number(filename):
        match = re.search(r'viewpoint_simulated_noise_(\d+)\.pcd', filename)
        return int(match.group(1)) if match else -1
    pcd_files = sorted(pcd_files, key=extract_number)
    
    # Track how many we saved for print logs
    count_hits = {name: 0 for name in feature_names}
        
    for noisy_pcd_path in pcd_files:
        viewpoint_name_noisy = os.path.basename(noisy_pcd_path).replace('.pcd', '')
        view_idx = viewpoint_name_noisy.replace('viewpoint_simulated_noise_', '')
        
        perfect_pcd_path = os.path.join(SIM_DIR, f"viewpoint_simulated_{view_idx}.pcd")
        if not os.path.exists(perfect_pcd_path):
            continue
            
        noisy_pcd = o3d.io.read_point_cloud(noisy_pcd_path)
        perfect_pcd = o3d.io.read_point_cloud(perfect_pcd_path)
        
        # Voronoi Segmentation for Noisy PCD
        dists_noisy = np.zeros((len(noisy_pcd.points), len(classes_pcd)))
        for i, cls_pcd in enumerate(classes_pcd):
            dists_noisy[:, i] = np.asarray(noisy_pcd.compute_point_cloud_distance(cls_pcd))
        assignments_noisy = np.argmin(dists_noisy, axis=1)
        
        # Voronoi Segmentation for Perfect PCD
        dists_perf = np.zeros((len(perfect_pcd.points), len(classes_pcd)))
        for i, cls_pcd in enumerate(classes_pcd):
            dists_perf[:, i] = np.asarray(perfect_pcd.compute_point_cloud_distance(cls_pcd))
        assignments_perf = np.argmin(dists_perf, axis=1)
        
        # Process each target feature (skip the background class which is the last one)
        for feature_idx in range(len(feature_names)):
            feature_name = feature_names[feature_idx]
            feature_cad = feature_pcds[feature_idx]
            
            # Extract Noisy Feature
            noisy_indices = np.where(assignments_noisy == feature_idx)[0]
            if len(noisy_indices) < 10:
                continue
            feature_noisy_pcd = noisy_pcd.select_by_index(noisy_indices)
            
            # Calculate Chamfer Distance (UNTRUNCATED!)
            dists_s2c = np.asarray(feature_noisy_pcd.compute_point_cloud_distance(feature_cad))
            dists_c2s = np.asarray(feature_cad.compute_point_cloud_distance(feature_noisy_pcd))
            if len(dists_c2s) == 0:
                continue
            chamfer_dist = np.mean(dists_s2c) + np.mean(dists_c2s)
            
            # Extract Perfect Feature
            perf_indices = np.where(assignments_perf == feature_idx)[0]
            if len(perf_indices) < 10:
                continue
            feature_perf_pcd = perfect_pcd.select_by_index(perf_indices)
            
            count_hits[feature_name] += 1
            
            # Export Perfect Feature for PointNet
            surface_clean_name = feature_name.replace('.stl', '')
            export_filename = f"viewpoint_simulated_{view_idx}_{surface_clean_name}.pcd"
            export_path = os.path.join(EXPORT_DIR, export_filename)
            o3d.io.write_point_cloud(export_path, feature_perf_pcd)
            
            # Save to CSV
            relative_path = f"{workpiece}/{export_filename}"
            results.append({
                "filename": relative_path,
                "dist_s2r": np.mean(dists_s2c),
                "dist_r2s": np.mean(dists_c2s),
                "chamfer_value": chamfer_dist,
                "asymmetry_value": 0.0
            })
            
    for feature_name, hits in count_hits.items():
        print(f"--> {feature_name}: Exported {hits} cropped point clouds")

print("\n==========================================")
print("DONE EXTRACTING FEATURES!")
print("==========================================")

df_results = pd.DataFrame(results)
CSV_EXPORT_DIR = f"processed_data/{DATASET_NAME}"
os.makedirs(CSV_EXPORT_DIR, exist_ok=True)
csv_path = os.path.join(CSV_EXPORT_DIR, "metadata.csv")
df_results.to_csv(csv_path, index=False)
print(f"Saved {len(df_results)} point cloud labels to {csv_path}")



ANALYZING & EXTRACTING SURFACES: TH0011AV
--> surface0.stl: Exported 432 cropped point clouds
--> surface1.stl: Exported 432 cropped point clouds
--> surface2.stl: Exported 432 cropped point clouds

ANALYZING & EXTRACTING SURFACES: TH0012AV
--> surface0.stl: Exported 432 cropped point clouds
--> surface1.stl: Exported 432 cropped point clouds
--> surface2.stl: Exported 432 cropped point clouds

ANALYZING & EXTRACTING SURFACES: TH0021AV
--> surface0.stl: Exported 432 cropped point clouds
--> surface1.stl: Exported 432 cropped point clouds
--> surface2.stl: Exported 432 cropped point clouds

ANALYZING & EXTRACTING SURFACES: TH0022AV
--> surface0.stl: Exported 216 cropped point clouds
--> surface1.stl: Exported 216 cropped point clouds
--> surface2.stl: Exported 216 cropped point clouds

ANALYZING & EXTRACTING SURFACES: TH0031AV
--> surface0.stl: Exported 216 cropped point clouds
--> surface1.stl: Exported 216 cropped point clouds
--> surface2.stl: Exported 216 cropped point clouds

ANAL

In [3]:
# ==========================================
# DEBUGGING VISUALIZATION FOR WORST INFERENCE ERRORS
# ==========================================
import math
import os
import glob
import numpy as np
import pandas as pd
import open3d as o3d
import re

# --- CONFIGURATION ---
RESULTS_CSV = r"C:\Users\Alvan\Documents\Alvan\Data\Code\Python\Main\pointnet_pytorch_reflective\data\5_simulation_2\inference_results_3_moe_extra.csv"
EXPERIMENT = "test_9_simulation_3"
NUMBER_OF_POINTS = 5000

# Change this number to view different errors! 
# 0 = the worst error, 1 = the second worst, 2 = the third worst, etc.
ERROR_RANK = 0 

print(f"Loading results from {RESULTS_CSV}...")
df = pd.read_csv(RESULTS_CSV)
df = df.sort_values(by="Error", ascending=False).reset_index(drop=True)

worst_case = df.iloc[ERROR_RANK]
filename = worst_case["Filename"]
gt_cd = worst_case["GroundTruth_CD"]
pred_cd = worst_case["Predicted_CD"]
error = worst_case["Error"]
surface_used = worst_case["Expert_Used"]

match = re.search(r'(TH\w+)/viewpoint_simulated_(\d+)_surface\d+', filename)
if not match:
    raise ValueError(f"Could not parse filename: {filename}")

WORKPIECE = match.group(1)
VIEWPOINT_IDX = int(match.group(2))

print(f"\n--- Debugging Error Rank {ERROR_RANK} ---")
print(f"File: {filename}")
print(f"Ground Truth CD: {gt_cd:.4f}")
print(f"Predicted CD:    {pred_cd:.4f}")
print(f"Absolute Error:  {error:.4f}")
print(f"\nLoading {WORKPIECE} - Viewpoint {VIEWPOINT_IDX} for interactive validation...")

PROCESSED_DIR = f"processed_data/{EXPERIMENT}/{WORKPIECE}"
SIM_DIR = f"viewpoints_candidate/testing_data/{EXPERIMENT}/{WORKPIECE}"
WORKPIECE_DIR = f"workpiece/{WORKPIECE}"

noisy_pcd_path = os.path.join(PROCESSED_DIR, f"viewpoint_simulated_noise_{VIEWPOINT_IDX}.pcd")
perfect_pcd_path = os.path.join(SIM_DIR, f"viewpoint_simulated_{VIEWPOINT_IDX}.pcd")
target_surface_path = os.path.join(WORKPIECE_DIR, f"{surface_used}.stl")

try:
    # ---------------------------------------------------------
    # TOP VISUALIZATION: Full CAD Feature (Red) & Full Noisy PCD (Gray)
    # ---------------------------------------------------------
    noisy_pcd = o3d.io.read_point_cloud(noisy_pcd_path)
    noisy_pcd.paint_uniform_color([0.5, 0.5, 0.5]) # Gray
    
    mesh = o3d.io.read_triangle_mesh(target_surface_path)
    mesh.compute_vertex_normals()
    feature_cad = mesh.sample_points_poisson_disk(number_of_points=NUMBER_OF_POINTS)
    feature_cad.paint_uniform_color([1.0, 0.0, 0.0]) # Red

    # ---------------------------------------------------------
    # BOTTOM VISUALIZATION: Raycasted Ideal (Green) & Raycasted Noisy (Blue)
    # ---------------------------------------------------------
    print("Running Voronoi Segmentation...")
    workpiece_mesh = o3d.io.read_triangle_mesh(os.path.join(WORKPIECE_DIR, "workpiece.stl"))
    workpiece_mesh.compute_vertex_normals()
    workpiece_pcd = workpiece_mesh.sample_points_poisson_disk(number_of_points=NUMBER_OF_POINTS * 4)

    surface_files = glob.glob(os.path.join(WORKPIECE_DIR, "surface*.stl"))
    feature_pcds = []
    target_feature_idx = -1

    for idx, f in enumerate(surface_files):
        m = o3d.io.read_triangle_mesh(f)
        m.compute_vertex_normals()
        p = m.sample_points_poisson_disk(number_of_points=NUMBER_OF_POINTS)
        feature_pcds.append(p)
        if surface_used in f:
            target_feature_idx = idx

    dists_to_features = np.ones(len(workpiece_pcd.points)) * 999.0
    for fpcd in feature_pcds:
        dists = np.asarray(workpiece_pcd.compute_point_cloud_distance(fpcd))
        dists_to_features = np.minimum(dists_to_features, dists)

    background_indices = np.where(dists_to_features > 2.0)[0]
    background_pcd = workpiece_pcd.select_by_index(background_indices)

    classes_pcd = feature_pcds.copy()
    if len(background_pcd.points) > 0:
        classes_pcd.append(background_pcd)

    # 1. Segment Noisy PCD
    dists_noisy = np.zeros((len(noisy_pcd.points), len(classes_pcd)))
    for i, cls_pcd in enumerate(classes_pcd):
        dists_noisy[:, i] = np.asarray(noisy_pcd.compute_point_cloud_distance(cls_pcd))
    assignments_noisy = np.argmin(dists_noisy, axis=1)

    noisy_indices = np.where(assignments_noisy == target_feature_idx)[0]
    feature_noisy_pcd = noisy_pcd.select_by_index(noisy_indices)
    feature_noisy_pcd.paint_uniform_color([0.0, 0.0, 1.0]) # Blue

    # 2. Segment Perfect PCD (Raycasted Ideal)
    perfect_pcd = o3d.io.read_point_cloud(perfect_pcd_path)
    dists_perf = np.zeros((len(perfect_pcd.points), len(classes_pcd)))
    for i, cls_pcd in enumerate(classes_pcd):
        dists_perf[:, i] = np.asarray(perfect_pcd.compute_point_cloud_distance(cls_pcd))
    assignments_perf = np.argmin(dists_perf, axis=1)

    perf_indices = np.where(assignments_perf == target_feature_idx)[0]
    feature_perf_pcd = perfect_pcd.select_by_index(perf_indices)
    feature_perf_pcd.paint_uniform_color([0.0, 1.0, 0.0]) # Green

    # Shift the Bottom Visualization down so they don't overlap
    # You can change this to [-100, 0, 0] if you prefer them side-by-side
    feature_perf_pcd.translate([0, 0, -100])
    feature_noisy_pcd.translate([0, 0, -100])

    geometries = [noisy_pcd, feature_cad, feature_perf_pcd, feature_noisy_pcd]

    def default_visualization(geoms, window_name):
        azimuth_deg = -45
        elevation_deg = -135
        az = math.radians(azimuth_deg)
        el = math.radians(elevation_deg)
        front = -np.array([math.cos(el) * math.cos(az), math.cos(el) * math.sin(az), math.sin(el)])
        lookat = geoms[0].get_center()
        up = [0, 0, 1]
        o3d.visualization.draw_geometries(geoms, window_name=window_name, width=1024, height=768, lookat=lookat, up=up, front=front)
    # 1st Window
    print("\nLaunching Window 1 (Full CAD & Full Noisy) ...")
    print("👉 CLOSE Window 1 to automatically open Window 2!")
    geometries_1 = [noisy_pcd, feature_cad]
    default_visualization(geometries_1, window_name=f"1. Full Scene - {filename} (Error: {error:.4f})")
    # 2nd Window
    print("\nLaunching Window 2 (Raycasted Ideal vs Raycasted Noisy) ...")
    geometries_2 = [feature_perf_pcd, feature_noisy_pcd]
    default_visualization(geometries_2, window_name=f"2. Cropped Features - {filename}")
except Exception as e:
    print(f"Error loading or visualizing point clouds: {e}")



Loading results from C:\Users\Alvan\Documents\Alvan\Data\Code\Python\Main\pointnet_pytorch_reflective\data\5_simulation_2\inference_results_3_moe_extra.csv...

--- Debugging Error Rank 0 ---
File: TH0042AV/viewpoint_simulated_281_surface1.pcd
Ground Truth CD: 11.3593
Predicted CD:    6.3966
Absolute Error:  4.9627

Loading TH0042AV - Viewpoint 281 for interactive validation...
Running Voronoi Segmentation...

Launching Window 1 (Full CAD & Full Noisy) ...
👉 CLOSE Window 1 to automatically open Window 2!

Launching Window 2 (Raycasted Ideal vs Raycasted Noisy) ...


In [44]:
# ==========================================
# INTERACTIVE DEBUGGING VISUALIZATION (TWO WINDOWS)
# ==========================================
import math
import os
import glob
import numpy as np
import open3d as o3d

# --- CONFIGURATION ---
WORKPIECE = "TH0011AV"
VIEWPOINT_IDX = 382
SURFACE_USED = "feature2"
EXPERIMENT = "test_9_simulation_3"
NUMBER_OF_POINTS = 5000

print(f"\nLoading {WORKPIECE} - Viewpoint {VIEWPOINT_IDX} for interactive validation...")

PROCESSED_DIR = f"processed_data/{EXPERIMENT}/{WORKPIECE}"
SIM_DIR = f"viewpoints_candidate/testing_data/{EXPERIMENT}/{WORKPIECE}"
WORKPIECE_DIR = f"workpiece/{WORKPIECE}"

noisy_pcd_path = os.path.join(PROCESSED_DIR, f"viewpoint_simulated_noise_{VIEWPOINT_IDX}.pcd")
perfect_pcd_path = os.path.join(SIM_DIR, f"viewpoint_simulated_{VIEWPOINT_IDX}.pcd")
target_surface_path = os.path.join(WORKPIECE_DIR, f"{SURFACE_USED}.stl")

try:
    # ---------------------------------------------------------
    # TOP VISUALIZATION: Full CAD Feature (Red) & Full Noisy PCD (Gray)
    # ---------------------------------------------------------
    noisy_pcd = o3d.io.read_point_cloud(noisy_pcd_path)
    if len(noisy_pcd.points) == 0:
        raise ValueError(f"Could not load noisy PCD from {noisy_pcd_path}")
    noisy_pcd.paint_uniform_color([0.5, 0.5, 0.5]) # Gray
    
    mesh = o3d.io.read_triangle_mesh(target_surface_path)
    mesh.compute_vertex_normals()
    feature_cad = mesh.sample_points_poisson_disk(number_of_points=NUMBER_OF_POINTS)
    feature_cad.paint_uniform_color([1.0, 0.0, 0.0]) # Red

    # ---------------------------------------------------------
    # BOTTOM VISUALIZATION: Raycasted Ideal (Green) & Raycasted Noisy (Blue)
    # ---------------------------------------------------------
    print("Running Voronoi Segmentation...")
    workpiece_mesh = o3d.io.read_triangle_mesh(os.path.join(WORKPIECE_DIR, "workpiece.stl"))
    workpiece_mesh.compute_vertex_normals()
    workpiece_pcd = workpiece_mesh.sample_points_poisson_disk(number_of_points=NUMBER_OF_POINTS * 4)

    surface_files = glob.glob(os.path.join(WORKPIECE_DIR, "feature*.stl"))
    feature_pcds = []
    target_feature_idx = -1

    for idx, f in enumerate(surface_files):
        m = o3d.io.read_triangle_mesh(f)
        m.compute_vertex_normals()
        p = m.sample_points_poisson_disk(number_of_points=NUMBER_OF_POINTS)
        feature_pcds.append(p)
        if SURFACE_USED in f:
            target_feature_idx = idx

    if target_feature_idx == -1:
        raise ValueError(f"Could not find surface file for {SURFACE_USED}")

    dists_to_features = np.ones(len(workpiece_pcd.points)) * 999.0
    for fpcd in feature_pcds:
        dists = np.asarray(workpiece_pcd.compute_point_cloud_distance(fpcd))
        dists_to_features = np.minimum(dists_to_features, dists)

    background_indices = np.where(dists_to_features > 2.0)[0]
    background_pcd = workpiece_pcd.select_by_index(background_indices)

    classes_pcd = feature_pcds.copy()
    if len(background_pcd.points) > 0:
        classes_pcd.append(background_pcd)

    # 1. Segment Noisy PCD
    dists_noisy = np.zeros((len(noisy_pcd.points), len(classes_pcd)))
    for i, cls_pcd in enumerate(classes_pcd):
        dists_noisy[:, i] = np.asarray(noisy_pcd.compute_point_cloud_distance(cls_pcd))
    assignments_noisy = np.argmin(dists_noisy, axis=1)

    noisy_indices = np.where(assignments_noisy == target_feature_idx)[0]
    feature_noisy_pcd = noisy_pcd.select_by_index(noisy_indices)
    feature_noisy_pcd.paint_uniform_color([1.0, 0.0, 0.0]) # Red

    # 2. Segment Perfect PCD (Raycasted Ideal)
    perfect_pcd = o3d.io.read_point_cloud(perfect_pcd_path)
    dists_perf = np.zeros((len(perfect_pcd.points), len(classes_pcd)))
    for i, cls_pcd in enumerate(classes_pcd):
        dists_perf[:, i] = np.asarray(perfect_pcd.compute_point_cloud_distance(cls_pcd))
    assignments_perf = np.argmin(dists_perf, axis=1)

    perf_indices = np.where(assignments_perf == target_feature_idx)[0]
    feature_perf_pcd = perfect_pcd.select_by_index(perf_indices)
    feature_perf_pcd.paint_uniform_color([0.0, 1.0, 0.0]) # Green

    # ---------------------------------------------------------
    # VISUALIZATION LAUNCHER
    # ---------------------------------------------------------
    def default_visualization(geoms, window_name):
        azimuth_deg = -45
        elevation_deg = -135
        az = math.radians(azimuth_deg)
        el = math.radians(elevation_deg)
        front = -np.array([math.cos(el) * math.cos(az), math.cos(el) * math.sin(az), math.sin(el)])
        lookat = geoms[0].get_center()
        up = [0, 0, 1]
        o3d.visualization.draw_geometries(geoms, window_name=window_name, width=1024, height=768, lookat=lookat, up=up, front=front, zoom = 0.8)

    # 1st Window
    print("\nLaunching Window 1 (Full CAD & Full Noisy) ...")
    print("👉 CLOSE Window 1 to automatically open Window 2!")
    geometries_1 = [noisy_pcd, feature_cad]
    # default_visualization(geometries_1, window_name=f"1. Full Scene - {WORKPIECE} View {VIEWPOINT_IDX}")

    # 2nd Window
    print("\nLaunching Window 2 (Raycasted Ideal vs Raycasted Noisy) ...")
    geometries_2 = [feature_perf_pcd, feature_noisy_pcd]
    default_visualization(geometries_2, window_name=f"2. Cropped Features - {WORKPIECE} View {VIEWPOINT_IDX}")

except Exception as e:
    print(f"Error loading or visualizing point clouds: {e}")



Loading TH0011AV - Viewpoint 382 for interactive validation...
Running Voronoi Segmentation...

Launching Window 1 (Full CAD & Full Noisy) ...
👉 CLOSE Window 1 to automatically open Window 2!

Launching Window 2 (Raycasted Ideal vs Raycasted Noisy) ...
